# NMS-free 推論

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/13-nms-free/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""The supervision must change before removing overlap suppression."""
import torch
from torch import nn
from torch.nn import functional as F


def iou(a, b):
    inter = (torch.minimum(a[2:], b[2:]) - torch.maximum(a[:2], b[:2])).clamp(min=0).prod()
    return inter / ((a[2:] - a[:2]).prod() + (b[2:] - b[:2]).prod() - inter)


def nms(boxes, scores, threshold=.5):
    # stable sort: equal scores keep input order, like the NMS in chapters 6 and 7
    order = scores.argsort(descending=True, stable=True).tolist()
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) <= threshold]
    return keep


def fit_scores(target):
    logits = nn.Parameter(torch.zeros(4))
    optimizer = torch.optim.SGD([logits], lr=1.)
    for _ in range(100):
        optimizer.zero_grad()
        loss = F.binary_cross_entropy_with_logits(logits, target)
        loss.backward()
        optimizer.step()
    return logits.sigmoid().detach()


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    boxes = torch.tensor([[0., 0., 10., 10.], [1., 0., 11., 10.], [20., 0., 30., 10.], [40., 0., 50., 10.]])
    many = fit_scores(torch.tensor([1., 1., 1., 0.]))
    one = fit_scores(torch.tensor([1., 0., 1., 0.]))
    many_ids = torch.where(many > .5)[0].tolist()
    one_ids = torch.where(one > .5)[0].tolist()
    kept = nms(boxes[many_ids], many[many_ids])
    many_after_nms = [many_ids[i] for i in kept]
    assert len(many_ids) == 3 and len(many_after_nms) == 2
    assert one_ids == [0, 2]
    assert 2 in many_after_nms and any(i in many_after_nms for i in (0, 1))
    manual_scores = torch.tensor([.92, .90, .80, .05])
    top2 = manual_scores.topk(2).indices.tolist()
    assert top2 == [0, 1]  # top-k itself cannot detect duplication
    # round in float64 so 0.959 prints as 0.959, not as float32's nearest value 0.9589999914...
    print('many-head scores:', many.double().round(decimals=3).tolist())
    print('one-head scores:', one.double().round(decimals=3).tolist())
    print('many/no NMS:', many_ids, 'many/NMS:', many_after_nms, 'one/no NMS:', one_ids)
    print('top-2 on duplicate-heavy scores:', top2, '(misses object at candidate 2)')
    print(f'duplicate IoU={iou(boxes[0], boxes[1]).item():.4f}')


if __name__ == '__main__':
    main()


many-head scores: [0.959, 0.959, 0.959, 0.041]
one-head scores: [0.959, 0.041, 0.959, 0.041]
many/no NMS: [0, 1, 2] many/NMS: [0, 2] one/no NMS: [0, 2]
top-2 on duplicate-heavy scores: [0, 1] (misses object at candidate 2)
duplicate IoU=0.8182
